<table style="width:100%">
<tr>
<td style="vertical-align:middle; text-align:left;">
<font size="2">
由 <a href="https://sebastianraschka.com">Sebastian Raschka</a> 撰写的 <a href="https://mng.bz/lZ5B">Build a Reasoning Model (From Scratch)</a> 一书配套的补充代码<br>
<br>代码仓库：<a href="https://github.com/rasbt/reasoning-from-scratch">https://github.com/rasbt/reasoning-from-scratch</a>
</font>
</td>
<td style="vertical-align:middle; text-align:left;">
<a href="https://mng.bz/lZ5B"><img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/cover-small.webp" width="100px"></a>
</td>
</tr>
</table>


# 附录 F：LLM 评估的常见方法

In [1]:
%pip install -r https://raw.githubusercontent.com/rasbt/reasoning-from-scratch/refs/heads/main/requirements.txt  # 安装项目依赖
from importlib.metadata import version  # 获取已安装库的版本信息

used_libraries = [
    "reasoning_from_scratch",  # 本项目主库
    "torch",                   # 深度学习库
    "tokenizers"               # 被项目依赖的分词器库
]

for lib in used_libraries:  # 遍历并打印版本号
    print(f"{lib} version: {version(lib)}")


reasoning_from_scratch version: 0.1.10
torch version: 2.9.0+cu126
tokenizers version: 0.22.1


&nbsp;
## F.1 理解 LLM 的主要评估方法

- 本节没有代码

<img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/appendix-f/Appendix_F_F01_raschka.webp" width="500px">

&nbsp;
### F.2 评估选项回答的准确率

<img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/appendix-f/Appendix_F_F02_raschka.webp" width="500px">

- 请注意，该图展示的是一种简化的多选题评估方式（如 MMLU），即将模型生成的答案字母与正确答案字母进行比对
- 实践中还有对数概率评分等变体，不再只看最终字母，而是计算模型认为每个候选答案的可能性
- 对于推理模型，这也可以包括在把正确答案输入模型时，它再次生成该答案的可能性
- 无论哪种方式，评估的核心都是检查模型是否选中了某个预先定义的答案
- （有关输出概率得分的更多内容，参见第 4 章对文本生成函数的改进）

&nbsp;
#### F.2.1 加载模型

In [2]:
from pathlib import Path                              # 处理跨平台文件路径
import torch                                          # 引入 PyTorch

from reasoning_from_scratch.ch02 import (             # 导入设备选择工具
    get_device
)
from reasoning_from_scratch.qwen3 import (            # 导入 Qwen3 下载、分词器、模型及配置
    download_qwen3_small,
    Qwen3Tokenizer,
    Qwen3Model,
    QWEN_CONFIG_06_B
)

device = get_device()                                 # 自动检测可用设备（CUDA/MPS/CPU）
torch.set_float32_matmul_precision("high")            # 设置浮点矩阵乘高精度

# 如果遇到兼容性问题，可取消下行注释强制使用 CPU
# device = "cpu"

WHICH_MODEL = "base"                                  # 选择加载的模型类型

if WHICH_MODEL == "base":                             # 基础版模型分支
    download_qwen3_small(                             # 下载基础版模型与分词器
        kind="base", tokenizer_only=False, out_dir="qwen3"
    )
    tokenizer_path = Path("qwen3") / "tokenizer-base.json"   # 基础版分词器路径
    model_path = Path("qwen3") / "qwen3-0.6B-base.pth"       # 基础版权重路径
    tokenizer = Qwen3Tokenizer(tokenizer_file_path=tokenizer_path)  # 创建分词器实例

elif WHICH_MODEL == "reasoning":                      # reasoning 版模型分支
    download_qwen3_small(                             # 下载 reasoning 版模型与分词器
        kind="reasoning", tokenizer_only=False, out_dir="qwen3"
    )
    tokenizer_path = Path("qwen3") / "tokenizer-reasoning.json"     # reasoning 版分词器路径
    model_path = Path("qwen3") / "qwen3-0.6B-reasoning.pth"         # reasoning 版权重路径
    tokenizer = Qwen3Tokenizer(                                 # 创建分词器实例，启用聊天模板与思考标志
        tokenizer_file_path=tokenizer_path,
        apply_chat_template=True,
        add_generation_prompt=True,
        add_thinking=True,
    )
else:
    raise ValueError(f"Invalid choice: WHICH_MODEL={WHICH_MODEL}")  # 非法模型名则报错

model = Qwen3Model(QWEN_CONFIG_06_B)                   # 根据配置构建模型
model.load_state_dict(torch.load(model_path))          # 加载预训练权重
model.to(device)                                       # 将模型移动到指定设备

USE_COMPILE = False                                    # 是否启用 torch.compile
if USE_COMPILE:
  torch._dynamo.config.allow_unspec_int_on_nn_module = True  # 允许 NN 模块中的未指定整型
  model = torch.compile(model)                         # 编译模型以潜在加速


Using NVIDIA CUDA GPU


/usr/local/lib/python3.12/dist-packages/torch/__init__.py:1617: UserWarning: Please use the new API settings to control TF32 behavior, such as torch.backends.cudnn.conv.fp32_precision = 'tf32' or torch.backends.cuda.matmul.fp32_precision = 'ieee'. Old settings, e.g, torch.backends.cuda.matmul.allow_tf32 = True, torch.backends.cudnn.allow_tf32 = True, allowTF32CuDNN() and allowTF32CuBLAS() will be deprecated after Pytorch 2.9. Please see https://pytorch.org/docs/main/notes/cuda.html#tensorfloat-32-tf32-on-ampere-and-later-devices (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:80.)
  _C._set_float32_matmul_precision(precision)


✓ qwen3/qwen3-0.6B-base.pth already up-to-date


&nbsp;
#### F.2.2 检查生成的答案字母

In [3]:
example = {                                             # 定义一道选择题样例
    "question": (                                       # 题干
        "How many ways are there to put 4 distinguishable"
        " balls into 2 indistinguishable boxes?"
    ),
    "choices": ["7", "11", "16", "8"],                  # 选项列表
    "answer": "D",                                      # 正确答案（选项字母）
}

def format_prompt(example):                             # 将样例格式化为提示字符串的函数
    return (
        f"{example['question']}\n"                      # 题干
        f"A. {example['choices'][0]}\n"                 # 选项 A
        f"B. {example['choices'][1]}\n"                 # 选项 B
        f"C. {example['choices'][2]}\n"                 # 选项 C
        f"D. {example['choices'][3]}\n"                 # 选项 D
        "Answer: "                                      # 结尾留空格，鼓励只生成一个选项字母
    )

prompt = format_prompt(example)                         # 生成提示字符串
print(prompt)                                           # 打印提示内容


How many ways are there to put 4 distinguishable balls into 2 indistinguishable boxes?
A. 7
B. 11
C. 16
D. 8
Answer: 


---


- 可以通过 `datasets` 库直接加载 MMLU 数据集中的示例（可用 `pip install datasets` 或 `uv add datasets` 安装）：

```python
from datasets import load_dataset

configs = get_dataset_config_names("cais/mmlu")
dataset = load_dataset("cais/mmlu", "high_school_mathematics")

# Inspect the first example from test set:
example = dataset["test"][0]
print(example)
```

- 上例使用 "high_school_mathematics" 子集；若想查看其他子集列表，可运行下方代码：


```python
from datasets import get_dataset_config_names

subsets = get_dataset_config_names("cais/mmlu")
print(subsets)
```

---


In [4]:
prompt_ids = tokenizer.encode(prompt)                     # 使用分词器将提示字符串编码为 token ID 列表
prompt_fmt = torch.tensor(prompt_ids, device=device).unsqueeze(0)  # 转为张量放到目标设备，并添加 batch 维度


- 我们生成少量 token，并提取模型首次输出的 A/B/C/D 字母：

In [5]:
from reasoning_from_scratch.ch02_ex import generate_text_basic_stream_cache  # 导入基础流式生成函数


def predict_choice(                                           # 根据提示生成并预测选项字母
    model, tokenizer, prompt_fmt, max_new_tokens=8
):
    pred = None                                               # 预测结果初始化为空
    for t in generate_text_basic_stream_cache(                # 流式逐 token 生成
        model=model,
        token_ids=prompt_fmt,
        max_new_tokens=max_new_tokens,
        eos_token_id=tokenizer.eos_token_id,
    ):
        answer = tokenizer.decode(t.squeeze(0).tolist())      # 解码当前生成的 token
        for letter in answer:                                 # 遍历解码出的字符
            letter = letter.upper()                           # 转为大写，兼容大小写
            if letter in "ABCD":                              # 如果字符是选项字母
                pred = letter                                 # 记录预测结果
                break                                         # 跳出字符遍历
        if pred:                                              # 已找到预测选项则停止生成
            break
    return pred                                               # 返回预测的选项字母


In [6]:
pred1 = predict_choice(model, tokenizer, prompt_fmt)  # 生成并预测选项字母

print(
    f"Generated letter: {pred1}\n"                   # 打印模型生成的选项
    f"Correct? {pred1 == example['answer']}"         # 判断是否与正确答案一致
)


Generated letter: C
Correct? False


&nbsp;
### F.3 使用验证器检查答案

- 本节没有代码（参见第 3 章）

<img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/appendix-f/Appendix_F_F03_raschka.webp" width="500px">

<br>
&nbsp;

### F.4 利用偏好和排行榜比较模型

<img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/appendix-f/Appendix_F_F04_raschka.webp" width="500px">

<img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/appendix-f/Appendix_F_F05_raschka.webp" width="500px">

- Elo 评分（"400 算法"）源自象棋排名：https://en.wikipedia.org/wiki/Performance_rating_(chess)
- 请注意，LM Arena 已改用统计的 Bradely-Terry 模型来给出类似 Elo 的分数，但依旧沿用成对排名的概念

In [ ]:
# 逐对“擂台赛”投票，元组格式：(胜者, 负者)
votes = [
    ("GPT-5", "Claude-3"),  # 第一场：GPT-5 优于 Claude-3
    ("GPT-5", "Llama-4"),   # 第二场：GPT-5 优于 Llama-4
    ("Claude-3", "Llama-3"),# 第三场：Claude-3 优于 Llama-3
    ("Llama-4", "Llama-3"), # 第四场：Llama-4 优于 Llama-3
    ("Claude-3", "Llama-3"),# 第五场：Claude-3 再次优于 Llama-3
    ("GPT-5", "Llama-3"),   # 第六场：GPT-5 优于 Llama-3
]


In [8]:
def elo_ratings(vote_pairs, k_factor=32, initial_rating=1000):
    # 初始化：所有模型的起始分相同
    ratings = {
        model: initial_rating
        for pair in vote_pairs
        for model in pair
    }

    # 按每场胜负依次更新 Elo
    for winner, loser in vote_pairs:

        # 根据当前分差计算获胜方的期望得分
        expected_winner = 1.0 / (
            1.0 + 10 ** ((ratings[loser] - ratings[winner]) / 400.0)
        )

        # k_factor 控制分数变动幅度
        ratings[winner] = (
            ratings[winner] + k_factor * (1 - expected_winner)
        )
        ratings[loser] = (
            ratings[loser] + k_factor * (0 - (1 - expected_winner))
        )

    return ratings  # 返回各模型最终 Elo 分


In [9]:
ratings = elo_ratings(votes, k_factor=32, initial_rating=1000)  # 计算各模型 Elo 分

for model in sorted(ratings, key=ratings.get, reverse=True):    # 按评分降序打印
    print(f"{model:8s} : {ratings[model]:.1f}")


GPT-5    : 1043.7
Claude-3 : 1015.2
Llama-4  : 1000.7
Llama-3  : 940.4


- 期望胜者得分的计算如下：

$$\text{expected\_winner} \;=\; \frac{1}{1 + 10^{\tfrac{\text{rating\_loser} - \text{rating\_winner}}{400}}}
$$

- 直观理解：
    - 若 rating_winner >> rating_loser：
       - 指数 → 非常负
       - 分母 ≈ 1
       - expected_winner ≈ 1（几乎稳赢）
    - 若 rating_winner << rating_loser：
       - 指数 → 非常正
       - 分母 → 很大
       - expected_winner ≈ 0（几乎必败）
    - 若 rating_winner == rating_loser：
       - 指数 = 0
       - 分母 = 2
       - expected_winner = 0.5（势均力敌）

&nbsp;
### F.5 使用其他 LLM 评判回答

<img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/appendix-f/Appendix_F_F06_raschka.webp" width="500px">

- 本节通过另一个更大的 LLM 来自动评估微调后 LLM 的回答
- 我们使用的是 Open AI 的 200 亿参数指令微调版 gpt-oss 模型，可通过 ollama ([https://ollama.com](https://ollama.com)) 在本地运行

- Ollama 是一个高效运行 LLM 的开源应用
- 它封装了 llama.cpp ([https://github.com/ggerganov/llama.cpp](https://github.com/ggerganov/llama.cpp))，该项目用纯 C/C++ 实现 LLM 以最大化效率
- 请注意，它是用于 LLM 推理（文本生成）的工具，而不是用于训练或微调 LLM
- 运行下方代码前，请访问 [https://ollama.com](https://ollama.com) 并按照说明（例如单击 "Download" 按钮并下载与你的操作系统匹配的 ollama 应用）完成安装

- macOS 和 Windows 用户请打开刚下载的 ollama 应用，若提示安装命令行用法，请选择 "yes"
- Linux 用户可以使用 ollama 官网提供的安装命令
- 在电脑上运行 ollama 的方式有 3 种：

**1. `ollama serve`**

- 以服务器形式运行 ollama 后端，通常监听 `http://localhost:11434`。只有当我们通过 API 调用时才会加载模型；如果想在 Python 中调用 ollama，就需要这种方式。

**2. `ollama run gpt-oss:20b`**

- 这是一个便捷封装。如果服务器尚未运行，它会先启动，然后（首次）下载模型，并进入可与模型聊天的交互式终端。其背后仍然使用同一个服务器 API。

**3. Ollama desktop app**

- 自动运行相同的后端并提供 GUI（如上图所示）。
它还会应用默认设置（系统提示、温度、停止序列），因此输出可能与直接调用 API 不同。

---

**注意**:

- 按照上述方式在终端中运行 `ollama serve` 时，可能会遇到 `Error: listen tcp 127.0.0.1:11434: bind: address already in use` 报错
- 若遇到该问题，可尝试运行 `OLLAMA_HOST=127.0.0.1:11435 ollama serve`（如果该地址仍被占用，可依次递增端口直到找到空闲地址）

---

- 例如，若想体验 ollama，可以运行 `ollama run gpt-oss:20b` 试用 200 亿参数的 gpt-oss 20B 模型。该模型（约 13 GB）会在第一次运行此命令时自动下载。（或者，可以像前图那样在桌面应用中使用。）
  
```bash
ollama run gpt-oss:20b
```


- 输出类似如下：

```
$ ollama run gpt-oss:20b
pulling manifest 
pulling b112e727c6f1: 100% ▕█████████████████████████████████▏  13 GB                         
pulling fa6710a93d78: 100% ▕█████████████████████████████████▏ 7.2 KB                         
pulling f60356777647: 100% ▕█████████████████████████████████▏  11 KB                         
pulling d8ba2f9a17b3: 100% ▕█████████████████████████████████▏   18 B                         
pulling 55c108d8e936: 100% ▕█████████████████████████████████▏  489 B                         
verifying sha256 digest 
writing manifest 
removing unused layers 
success
```

- 如需了解 gpt-oss 的更多细节，可阅读我的长文 [From GPT-2 to gpt-oss: Analyzing the Architectural Advances](https://magazine.sebastianraschka.com/p/from-gpt-2-to-gpt-oss-analyzing-the) 
- 使用 "gpt-oss:20b" 模型（20B 参数）需要 13 GB RAM；如果机器无法满足，可尝试更小的 `qwen3:4b` 模型（约需 4 GB RAM）
- 如果硬件允许，也可以改用更大的 120B gpt-oss (`qwen3:235b`) 甚至 235B 的 Qwen3 (`qwen3:235b`)
- 下载完成后，会看到一个可以与模型对话的命令行提示符
- 试着输入 "What is 1+2?"，模型会返回类似下述的输出

```
>>> What is 1+2?
Thinking...
User asks: "What is 1+2?" This is simple: answer 3. Provide explanation? Possibly ask for simple 
arithmetic. Provide answer: 3.
...done thinking.

1 + 2 = **3**
```


- 输入 `/bye` 可结束该会话

- 下方代码会在使用 ollama 评估上一节生成的测试集回答之前，先检查 ollama 会话是否正常运行

In [ ]:
import psutil  # 进程监控库，用于枚举正在运行的进程

def check_if_running(process_name):
    running = False                          # 标记是否找到目标进程
    for proc in psutil.process_iter(["name"]):  # 遍历所有进程，仅取 name 属性
        if process_name in proc.info["name"]:   # 如果进程名包含目标字符串
            running = True                      # 标记为已运行
            break                               # 已找到即可退出循环
    return running                              # 返回是否在运行

ollama_running = check_if_running("ollama")     # 检查名中包含 “ollama” 的进程是否存在

if not ollama_running:                          # 如果未运行则抛出异常提醒
    raise RuntimeError(
        "Ollama not running. Launch ollama before proceeding."
    )
print("Ollama running:", check_if_running("ollama"))  # 再次检查并打印状态


- 除了之前使用的 `ollama run` 命令外，还可以通过其 REST API 在 Python 中调用模型，代码如下函数
- 在运行本笔记本后续的单元格前，请确保 ollama 仍在运行（前面的代码单元应打印 `"Ollama running: True"`）
- 接着运行下一个代码单元向模型发起查询

In [ ]:
import json                                  # 处理 JSON 序列化/反序列化
import requests                              # 发送 HTTP 请求

def query_model(
    prompt,
    model="gpt-oss:20b",
    # 如果用 OLLAMA_HOST=127.0.0.1:11435 启动了服务，记得把端口改成 11435
    url="http://localhost:11434/api/chat"
):
    # 构造请求体
    data = {
        "model": model,                      # 指定要调用的模型名称
        "messages": [
            {"role": "user", "content": prompt}  # 用户输入内容
        ],
        "options": {                         # 下面设置保证输出确定性
            "seed": 123,                     # 固定随机种子
            "temperature": 0,                # 温度设为 0（贪心/确定性）
            "num_ctx": 2048                  # 上下文窗口大小
        }
    }

    # 发送 POST 请求（流式读取响应）
    with requests.post(url, json=data, stream=True, timeout=30) as r:
        r.raise_for_status()                 # 请求失败直接抛异常
        response_data = ""                   # 用于累积返回文本
        for line in r.iter_lines(decode_unicode=True):  # 逐行消费流式响应
            if not line:                     # 跳过空行
                continue
            response_json = json.loads(line) # 每行是一个 JSON 块
            if "message" in response_json:   # 如果包含消息内容
                response_data += response_json["message"]["content"]  # 追加到最终字符串

    return response_data                      # 返回模型完整回复


In [ ]:
ollama_model = "gpt-oss:20b"                          # 指定要调用的 Ollama 模型
result = query_model("What is 1+2?", ollama_model)    # 调用上面封装的 query_model 发送请求
print(result)                                         # 打印模型返回的结果


- 现在，可以使用上面定义的 `query_model` 函数来评估我们自己模型的回答

In [ ]:
def rubric_prompt(instruction, reference_answer, model_answer):
    # 评分准则说明，描述 1~5 的评分标准
    rubric = (
        "You are a fair judge assistant. You will be given an instruction, "
        "a reference answer, and a candidate answer to evaluate, according "
        "to the following rubric:\n\n"
        "1: The response fails to address the instruction, providing "
        "irrelevant, incorrect, or excessively verbose content.\n"
        "2: The response partially addresses the instruction but contains "
        "major errors, omissions, or irrelevant details.\n"
        "3: The response addresses the instruction to some degree but is "
        "incomplete, partially correct, or unclear in places.\n"
        "4: The response mostly adheres to the instruction, with only "
        "minor errors, omissions, or lack of clarity.\n"
        "5: The response fully adheres to the instruction, providing a "
        "clear, accurate, and relevant answer in a concise and efficient "
        "manner.\n\n"
        "Now here is the instruction, the reference answer, and the "
        "response.\n"
    )

    # 拼接最终提示词：包含指令、参考答案与待评答案
    prompt = (
        f"{rubric}\n"
        f"Instruction:\n{instruction}\n\n"
        f"Reference Answer:\n{reference_answer}\n\n"
        f"Answer:\n{model_answer}\n\n"
        f"Evaluation: "
    )
    return prompt


- `model_answer` 可以是我们自己模型生成的答案；此处为了简单起见，先硬编码了一个示例回答

In [ ]:
rendered_prompt = rubric_prompt(
    instruction=(
        "If all birds can fly, and a penguin is a bird, "
        "can a penguin fly?"
    ),
    reference_answer=(
        "Yes, according to the premise that all birds can fly, "
        "a penguin can fly."
    ),
    model_answer=(
        "Yes – under those premises a penguin would be able to fly."
    )
)
print(rendered_prompt)  # 打印拼接好的评分提示词


In [ ]:
result = query_model(rendered_prompt, ollama_model)  # 调用本地 Ollama 模型对评分提示词进行推理
print(result)                                       # 打印模型返回的评分结果
